In [10]:
import pandas as pd
import numpy as np
import ml_utils as mt
import seaborn as sns

In [29]:
import pandas as pd
data = pd.read_csv("loan_data_train.csv")

In [12]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2200 entries, 0 to 2199
Data columns (total 15 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   ID                              2199 non-null   float64
 1   Amount.Requested                2199 non-null   object 
 2   Amount.Funded.By.Investors      2199 non-null   object 
 3   Interest.Rate                   2200 non-null   object 
 4   Loan.Length                     2199 non-null   object 
 5   Loan.Purpose                    2199 non-null   object 
 6   Debt.To.Income.Ratio            2199 non-null   object 
 7   State                           2199 non-null   object 
 8   Home.Ownership                  2199 non-null   object 
 9   Monthly.Income                  2197 non-null   float64
 10  FICO.Range                      2200 non-null   object 
 11  Open.CREDIT.Lines               2196 non-null   object 
 12  Revolving.CREDIT.Balance        21

In [13]:
#ignore : id [unique id], Interest.Rate[target]
###info : numeric: data type : object :typecasting
#Amount.requested
#amount.Funded.By.Investors


### info :categorical , create dummies for it
# Loan.length
# FICO.Range : custom function
value = data['Loan.Purpose'].value_counts(dropna = False)
value

Loan.Purpose
debt_consolidation    1147
credit_card            394
other                  174
home_improvement       135
major_purchase          84
small_business          80
car                     45
wedding                 35
medical                 26
moving                  25
house                   19
vacation                18
educational             14
renewable_energy         3
NaN                      1
Name: count, dtype: int64

In [14]:
import ml_utils as mt

In [15]:
custom_func_cols={}

In [16]:
a = data['Debt.To.Income.Ratio']

In [17]:
temp = a.str.replace('%','')

In [18]:
num = pd.to_numeric(temp,errors='coerce')
num

0       27.56
1       13.39
2        3.50
3       19.62
4       23.79
        ...  
2195    12.10
2196    14.16
2197    15.03
2198    11.63
2199     3.83
Name: Debt.To.Income.Ratio, Length: 2200, dtype: float64

In [19]:
def custom_dir(dir_col):
    temp = a.str.replace('%','')
    num = pd.to_numeric(temp,errors='coerce')
    return num

In [20]:
custom_dir(data['Debt.To.Income.Ratio'])

0       27.56
1       13.39
2        3.50
3       19.62
4       23.79
        ...  
2195    12.10
2196    14.16
2197    15.03
2198    11.63
2199     3.83
Name: Debt.To.Income.Ratio, Length: 2200, dtype: float64

In [21]:
b = data['FICO.Range']

In [22]:
def custom_FICO(b):
    k = b.str.split('-',expand = True)
    for i in [0,1]:
        k[i]= pd.to_numeric(k[i],errors='coerce')
    n = 0.5*(k[0]+k[1])
    return n

In [23]:
custom_FICO(data['FICO.Range'])

0       722.0
1       712.0
2       692.0
3       712.0
4       732.0
        ...  
2195    677.0
2196    702.0
2197    677.0
2198    672.0
2199    712.0
Length: 2200, dtype: float64

In [24]:
a= data['Employment.Length']

In [25]:
def custom_el(el_col):
    temp=el_col.replace({'5 years':5,'4 years':4,'<1 year':0,'10+ years':10,'2 years':2,'3 years':3,'1 year':1,'6 years':6,'7 years':7,'8 years':8,'9 years':9})
    num = pd.to_numeric(temp,errors='coerce')
    return num

In [31]:
cat_to_num_cols=['Amount.Requested','Open.CREDIT.Lines','Revolving.CREDIT.Balance']
simple_num_cols=['Monthly.Income','Inquiries.in.the.Last.6.Months']
cat_to_dummies_cols=['Loan.Length','Loan.Purpose','State','Home.Ownership']
cust_func_cols={'Debt.To.Income.Ratio':custom_dir,'FICO.Range':custom_FICO,'Employment.Length':custom_el}

In [32]:
ld_pipe=mt.DataPipe(simple_num=simple_num_cols,cat_to_dummies=cat_to_dummies_cols,cat_to_num=cat_to_num_cols,custom_func_dict=custom_func_cols)

In [33]:
ld_pipe.fit(data)

d:\prog\Python(Narind Verma)\WORKSHOP\w1\ml_utils.py:75: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  x[col]=pd.to_numeric(x[col],errors='coerce')


In [37]:
x_train = ld_pipe.transform(data)

d:\prog\Python(Narind Verma)\WORKSHOP\w1\ml_utils.py:75: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  x[col]=pd.to_numeric(x[col],errors='coerce')


In [38]:
y_train = data['Interest.Rate'].str.replace('%','').astype(float)

In [39]:
from sklearn .linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

In [40]:
lr = LinearRegression()

In [41]:
scores = cross_val_score(lr,x_train,y_train,cv =10,scoring = 'neg_mean_absolute_error')

In [42]:
scores.mean()

-2.9274014660161582

In [43]:
lr.fit(x_train,y_train)

LinearRegression()

In [44]:
lr.intercept_

12.19283352002364

In [45]:
list(zip(x_train.columns,lr.coef_))

[('Loan.Length_36 months', 1.9981232786154637),
 ('Loan.Length_60 months', 5.534997751643802),
 ('Loan.Purpose_debt_consolidation', -1.039756743665145),
 ('Loan.Purpose_credit_card', -0.9984901266967061),
 ('Loan.Purpose_other', -0.8549061315421832),
 ('Loan.Purpose_home_improvement', -2.8218738773144234),
 ('Loan.Purpose_major_purchase', -2.77504760187266),
 ('Loan.Purpose_small_business', -1.763622143203787),
 ('Loan.Purpose___other__', -1.9263424041209234),
 ('Loan.Purpose_car', -3.2362610704120582),
 ('Loan.Purpose_wedding', -1.3064204065598843),
 ('Loan.Purpose_medical', -2.0524268893732995),
 ('State_CA', -0.4851802873884673),
 ('State_NY', -0.5153263164587579),
 ('State___other__', -0.00762041838392096),
 ('State_FL', -0.1365306139897254),
 ('State_TX', 0.16509372243308493),
 ('State_PA', -0.7797416329297414),
 ('State_IL', -1.3576419328318765),
 ('State_GA', -0.6110377635145199),
 ('State_NJ', -0.593810295822028),
 ('State_VA', -0.33324878143406145),
 ('State_MA', -1.0097712049

In [47]:
effect_df = x_train*lr.coef_

In [48]:
effect_df.mean()

Loan.Length_36 months              1.563986
Loan.Length_60 months              1.197572
Loan.Purpose_debt_consolidation   -0.542091
Loan.Purpose_credit_card          -0.178821
Loan.Purpose_other                -0.067615
Loan.Purpose_home_improvement     -0.173160
Loan.Purpose_major_purchase       -0.105956
Loan.Purpose_small_business       -0.064132
Loan.Purpose___other__            -0.048159
Loan.Purpose_car                  -0.066196
Loan.Purpose_wedding              -0.020784
Loan.Purpose_medical              -0.024256
State_CA                          -0.082922
State_NY                          -0.054109
State___other__                   -0.000755
State_FL                          -0.009247
State_TX                           0.010956
State_PA                          -0.031190
State_IL                          -0.053689
State_GA                          -0.022775
State_NJ                          -0.021863
State_VA                          -0.010603
State_MA                        